In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [23]:
prompt=PromptTemplate.from_template(""" Your are an ai instructor explain {topic} to {audience}
 requirements:
 use simple professional english
include one practical example
keep the answer under {word_limit} words
""")

In [24]:
model = ChatGroq(
    model="groq/compound-mini"
)

In [25]:
chain=prompt|model

In [26]:
response = chain.invoke({
    "topic": "Langchain",
    "audience": "beginners",
    "word_limit": 80
})

print(response.content)

APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
prompting = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced AI instructor. Use simple English."),
    ("human", "Explain {topic} to {audience}. Include one example.")
])

In [ ]:
chaining=prompting|model
response=chaining.invoke({
    "topic": "Langchain",
    "audience": "beginners",
    "word_limit": 80
})
print(response.content)

**What is LangChain?**

LangChain is a **software library** that helps developers build applications that use large language models (LLMs) like ChatGPT, Claude, or Gemini.  
Think of it as a set of building blocks (or “chains”) that let you:

1. **Talk to an LLM** – send a prompt and get a response.  
2. **Add extra logic** – decide what to ask, how to format the answer, or what to do with the answer.  
3. **Connect to other tools** – databases, APIs, web search, file storage, etc.  

By chaining these pieces together, you can create more powerful and useful AI apps than a simple “type‑and‑receive‑answer” chatbot.

---

### Key Ideas (in simple terms)

| Concept | What it means | Why it helps |
|---------|---------------|--------------|
| **PromptTemplate** | A reusable text pattern with blanks you fill in. | Keeps prompts consistent and easy to change. |
| **LLMChain** | A “chain” that takes a prompt, sends it to the LLM, and returns the result. | Encapsulates the ask‑and‑reply step. 

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are an AI instructor. Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}

{% if level == "beginner" %}
Use simple professional English and avoid complex terms.

{% elif level == "intermediate" %}
Use professional English and include some technical terms.

{% else %}
Use moderate technical depth.
{% endif %}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["topic", "audience", "include_example", "level"],
    template_format="jinja2"
)


In [ ]:
final_answer= prompt.invoke({
    "topic": "RAG",
    "audience": "beginners",
    "include_example": True,
    "level": "beginner"
})

In [ ]:
#prompt_management
import json
with open("prompt.json", "r") as file:
    prompts=json.load(file)

In [ ]:
prompt=prompts["rag_prompt"]
final_prompt=prompt.format(
    context="Employee receive 24 paid leaves every year",
    question="How many paid leaves can an employee take in 2 years?"
)
print(final_prompt)

Answer the question using only the provided context.

Context: Employee receive 24 paid leaves every year

Question: How many paid leaves can an employee take in 2 years?


In [28]:
def prompt_load(prompt_name):
    with open("prompt.json","r")as file:
        prompts=json.load(file)
        config=prompts[prompt_name]
        messages=[
            (message["role"], message["template"]) for message in config["messages"]
        ]
        final_prompt=ChatPromptTemplate.from_messages(messages,template_format=config["template_format"])
        return final_prompt

In [29]:
prompt=prompt_load("rag_prompt")

In [32]:
result=prompt.invoke({
    "context": "Employee receive 24 paid leaves every year",
    "question": "How many paid leaves can an employee take in 2 years?"
})
for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM
You are a RAG assistant. Answer only from the provided context. If the answer is not available, say 'I do not have enough information.'
HUMAN
Context:
Employee receive 24 paid leaves every year

Question:
How many paid leaves can an employee take in 2 years?
